# Chapter 10. Building Neural Networks with PyTorch

PyTorch is a deep learning library for building and training neural networks.

Like NumPy, it provides multidimensional arrays for numerical computation. In addition, it supports automatic differentiation and hardware acceleration.

## PyTorch Fundamentals

The core data structure in PyTorch is the tensor.

A tensor is a multidimensional array with a shape and data type, similar to a NumPy array. Unlike NumPy arrays, tensors can use hardware accelerators and support automatic differentiation.

### PyTorch Tensors

In [6]:
import torch

X = torch.tensor([[1.0, 4.0, 7.0], [2.0, 3.0, 6.0]])

print(X)

tensor([[1., 4., 7.],
        [2., 3., 6.]])


Like NumPy arrays, tensors have a shape and a data type.

In [13]:
print("Shape:", X.shape)
print("\nData type:", X.dtype)

Shape: torch.Size([2, 3])

Data type: torch.float32


Tensor indexing and slicing work similarly to NumPy.

In [22]:
print(X[0, 1])
print(X[:, 1])

tensor(4.)
tensor([4., 3.])


PyTorch supports elementwise operations, reductions, and matrix multiplication.

In [27]:
10 * (X + 1.0) # itemwise addition and multiplication

tensor([[20., 50., 80.],
        [30., 40., 70.]])

In [29]:
X.exp() # itemwise exponential

tensor([[   2.7183,   54.5981, 1096.6332],
        [   7.3891,   20.0855,  403.4288]])

In [31]:
X.mean()

tensor(3.8333)

In [35]:
X.max(dim=0) # max values along dimension 0 (i.e., max value per column)

torch.return_types.max(
values=tensor([2., 4., 7.]),
indices=tensor([1, 0, 0]))

In [37]:
X @ X.T # matrix transpose and matrix multiplication

tensor([[66., 56.],
        [56., 49.]])

Tensors can be converted to NumPy arrays, and NumPy arrays can be converted to tensors.

PyTorch uses `float32` by default for floating-point tensors, while NumPy usually uses `float64`.

In [54]:
import numpy as np

print("Tensor converted to a NumPy array:")
X.numpy()

Tensor converted to a NumPy array:


array([[1., 4., 7.],
       [2., 3., 6.]], dtype=float32)

In [58]:
print("Tensor created from a NumPy array:")
torch.tensor(np.array([[1., 4., 7.], [2., 3., 6.]]))

Tensor created from a NumPy array:


tensor([[1., 4., 7.],
        [2., 3., 6.]], dtype=torch.float64)

Using `float32` is common in deep learning because it uses less memory and is generally faster than `float64`, while providing enough precision for most neural network tasks.

PyTorch supports in-place operations, which modify an existing tensor directly.

These operations end with an underscore, such as `relu_()` or `zero_()`.

In [68]:
X[:, 1] = -99

print("After modifying the second column:")
X

After modifying the second column:


tensor([[  1., -99.,   7.],
        [  2., -99.,   6.]])

In [70]:
X.relu_()

print("After applying ReLU in place:")
X

After applying ReLU in place:


tensor([[1., 0., 7.],
        [2., 0., 6.]])

In-place operations can save memory, but they should be used carefully when working with autograd because modifying tensors needed for backpropagation can cause errors.

### Hardware Acceleration

PyTorch tensors can run on GPUs and other supported accelerators.

Using a GPU can greatly speed up large matrix operations and neural network training.

The following code checks whether an NVIDIA GPU is available through CUDA. If not, it checks for Apple's Metal Performance Shaders (MPS). Otherwise, it uses the CPU.

In [77]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Selected device:", device)

Selected device: cuda


A tensor can be moved to the selected device using `.to(device)`.

Once a tensor is on a GPU, operations involving that tensor will also run on the GPU.

In [85]:
M = torch.tensor([[1., 2., 3.], [4., 5., 6.]])

M = M.to(device)

print("Tensor device:")
print(M.device)

Tensor device:
cuda:0


Alternatively, a tensor can be created directly on the selected device.

In [89]:
M = torch.tensor([[1., 2., 3.], [4., 5., 6.]], device=device)

print("Tensor device:")
print(M.device)

Tensor device:
cuda:0


In [93]:
R = M @ M.T # run some operations on the GPU

print("Matrix multiplication result:")
print(R)

print("\nResult device:")
print(R.device)

Matrix multiplication result:
tensor([[14., 32.],
        [32., 77.]], device='cuda:0')

Result device:
cuda:0


Keeping tensors and model parameters on the same device avoids repeated CPU-to-GPU transfers.

Data transfer can become a bottleneck, especially when the model is small or the batches are too small.

GPUs are most effective for large parallel computations.

For small operations, the overhead of moving data and coordinating GPU work can reduce or eliminate the speed advantage.

In [97]:
M = torch.rand((1000, 1000)) # on the CPU

%timeit M @ M.T

13.2 ms ± 165 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [99]:
M = torch.rand((1000, 1000), device="cuda") # on the GPU

%timeit M @ M.T

126 μs ± 1.47 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


### Autograd

PyTorch's autograd automatically tracks tensor operations and computes gradients during backpropagation.

To track gradients, create tensors with `requires_grad=True`.

In [118]:
x = torch.tensor(5.0, requires_grad=True)

f = x ** 2

print("f:")
print(f)

f:
tensor(25., grad_fn=<PowBackward0>)


In [120]:
f.backward()

print("Gradient of f with respect to x:")
print(x.grad)

Gradient of f with respect to x:
tensor(10.)


For $f(x) = x^2$, autograd computes:

$$
\frac{df}{dx} = 2x
$$

At $x = 5$, the gradient is $10$.

`backward()` performs the backward pass and stores gradients in the `.grad` attribute of tracked tensors.

In [124]:
learning_rate = 0.1

with torch.no_grad():
    x -= learning_rate * x.grad # gradient descent step

print("Updated x:")
print(x)

Updated x:
tensor(4., requires_grad=True)


Use `torch.no_grad()` when updating parameters or making predictions.

This prevents PyTorch from tracking these operations in the computation graph.

In [127]:
x.grad.zero_()

print("Gradient after reset:")
print(x.grad)

Gradient after reset:
tensor(0.)


PyTorch accumulates gradients by default.

Therefore, gradients should be reset before the next backward pass.

In [132]:
learning_rate = 0.1

x = torch.tensor(5.0, requires_grad=True)

for iteration in range(100):
    f = x ** 2 # forward pass
    f.backward() # backward pass
    with torch.no_grad():
        x -= learning_rate * x.grad

    x.grad.zero_() # reset the gradients

print("Final x:")
print(x)

Final x:
tensor(1.0185e-09, requires_grad=True)


Each iteration follows this pattern:

1. Forward pass: compute the output and loss
2. Backward pass: compute gradients with `backward()`
3. Update parameters
4. Reset gradients

The `zero_()` method is an in-place operation, meaning that it modifies the existing gradient tensor directly.

In-place operations end with `_` and can save memory, but they should be used carefully because modifying a tensor needed for backpropagation can cause a `RuntimeError`.